In [14]:
# Importing libraries
import itertools
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from mp_api.client import MPRester
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from pymatgen.core import Lattice, Structure
from pymatgen.transformations.standard_transformations import OrderDisorderedStructureTransformation
from pymatgen.io.cif import CifWriter
warnings.filterwarnings("ignore")

In [ ]:
# Data harvesting for three categories of HEA (Refractoriness, Corrosion resistant and Lightweight)
import os
from dotenv import load_dotenv
from mp_api.client import MPRester
import warnings
warnings.filterwarnings("ignore")

# Loading environment variables
load_dotenv()
api_key = os.getenv("MP_API_KEY")

# Combine the elements and their Valence Electron Concentration
alloy_category= {
    "Refractory":  {'W': 6, 'Mo': 6, 'Ta': 5, 'Nb': 5, 'V': 5},
    "Corrosion":   {'Co': 9, 'Cr': 6, 'Fe': 8, 'Ni': 10, 'Cu': 11},
    "Lightweight": {'Al': 3, 'Mg': 2, 'Li': 1, 'Ti': 4, 'Zn': 12}
}

master_props = {"Refractory": {}, "Corrosion": {}, "Lightweight": {}}

print("Starting data Harvesting...")

with MPRester(api_key) as mpr:
    for family_name, elements_dict in alloy_category.items():
        print(f"\nHarvesting {family_name} elements...")
        
        # Identifying element symbols and their VEC
        for el_symbol, true_vec in elements_dict.items():
            try:
                docs = mpr.materials.summary.search(elements=[el_symbol], is_stable=True)
                if docs:
                    struct = docs[0].structure
                    
                    # Storing the VEC
                    master_props[family_name][el_symbol] = {
                        'r': struct.species[0].atomic_radius,
                        'vec': true_vec, 
                        'lattice_const': struct.lattice.a,
                        'density': struct.density
                    }
                    print(f"  [+] {el_symbol} harvested. (VEC: {true_vec})")
            except Exception as e:
                print(f"  [-] Error fetching {el_symbol}: {e}")

print("\nData harvesting complete.")

Starting data Harvesting...

Harvesting Refractory elements...


Retrieving SummaryDoc documents: 100%|██████████| 478/478 [00:00<00:00, 1600061.70it/s]


  [+] W harvested. (VEC: 6)


Retrieving SummaryDoc documents: 100%|██████████| 867/867 [00:00<00:00, 1232698.84it/s]


  [+] Mo harvested. (VEC: 6)


...

  [+] Li harvested. (VEC: 1)


Retrieving SummaryDoc documents: 100%|██████████| 990/990 [00:00<00:00, 8304721.92it/s]


  [+] Ti harvested. (VEC: 4)


Retrieving SummaryDoc documents: 100%|██████████| 1279/1279 [00:03<00:00, 328.95it/s]

  [+] Zn harvested. (VEC: 12)

Data harvesting complete. Clean, accurate 'master_props' loaded in memory.


In [19]:
# For Refractory
import itertools
import numpy as np
import pandas as pd

# Percentage steps from 5% to 35% according Wikipedia
allowed_percentages = range(5, 40, 5)

# Storing the filtered alloys for each family of category
stable_candidates = {
    "Refractory": pd.DataFrame(),
    "Corrosion": pd.DataFrame(),
    "Lightweight": pd.DataFrame()
}

print("Starting Combinatorial Engine Using Itertools...\n")

for family_name, elements in alloy_families.items():
    print(f"Processing {family_name}...")
    
    # Calcuting all combinations for the specific category
    valid_compositions = []
    for combo in itertools.product(allowed_percentages, repeat=len(elements)):
        if sum(combo) == 100:
            comp = dict(zip(elements, [x/100.0 for x in combo]))
            valid_compositions.append(comp)
            
    # Calculating Average lattice mismatch(Delta) and Valence Electron Concentration (VEC)
    results = []
    for comp in valid_compositions:
        vec_total = sum(frac * master_props[family_name][el]['vec'] for el, frac in comp.items())
        r_avg = sum(frac * master_props[family_name][el]['r'] for el, frac in comp.items())
        variance_sum = sum(frac * (1 - master_props[family_name][el]['r'] / r_avg)**2 for el, frac in comp.items())
        delta = 100 * np.sqrt(variance_sum)
        
        # Calculating theoretical density specifically for the Lightweight group
        density = sum(frac * master_props[family_name][el]['density'] for el, frac in comp.items())
        
        results.append({**comp, 'VEC': round(vec_total, 3), 'Delta': round(delta, 3), 'Density': round(density, 2)})
        
    df_results = pd.DataFrame(results)
    
    # The family specific stability filters
    if family_name == "Refractory":
        # BCC Rule: Delta < 6.6 AND 5.0 <= VEC <= 6.8
        df_stable = df_results[(df_results['Delta'] < 6.6) & (df_results['VEC'] >= 5.0) & (df_results['VEC'] <= 6.8)]
    
    elif family_name == "Corrosion":
        # FCC Rule: Delta < 6.6 AND VEC >= 8.0
        df_stable = df_results[(df_results['Delta'] < 6.6) & (df_results['VEC'] >= 8.0)]
        
    elif family_name == "Lightweight":
        # Mixed/HCP Rule: Using just strict lattice strain (Delta < 6.6)
        df_stable = df_results[(df_results['Delta'] < 6.6)]

    stable_candidates[family_name] = df_stable.copy()
    
    print(f"  -> Generated {len(df_results)} theoretically possible HEAs.")
    print(f"  -> {len(df_stable)} survived the {family_name} stability filter.\n")

print("All processed successfully!")

Starting Combinatorial Engine Using Itertools...

Processing Refractory...
  -> Generated 1451 theoretically possible HEAs.
  -> 1451 survived the Refractory stability filter.

Processing Corrosion...
  -> Generated 1451 theoretically possible HEAs.
  -> 1428 survived the Corrosion stability filter.

Processing Lightweight...
  -> Generated 1451 theoretically possible HEAs.
  -> 1099 survived the Lightweight stability filter.

All processed successfully!


In [ ]:
# Generating cell
import random
from pymatgen.core import Lattice, Structure, Species
from pymatgen.io.cif import CifWriter
import warnings

warnings.filterwarnings("ignore")

print("\n[4] Creating 3D physical blueprint...")

TOTAL_ATOMS = 54

# Calculating how many whole atoms each element should get
atom_counts = {el: int(round(best_alloy[el] * TOTAL_ATOMS)) for el in elements}

difference = TOTAL_ATOMS - sum(atom_counts.values())
if difference != 0:
    max_el = max(atom_counts, key=atom_counts.get)
    atom_counts[max_el] += difference

print(f"Discrete Atom Mapping for {TOTAL_ATOMS}-atom cell:")
for el, count in atom_counts.items():
    print(f"  {el}: {count} atoms")

# Creating the list of 54 neutral atoms
atom_list = []
for el, count in atom_counts.items():
    atom_list.extend([Species(el, 0)] * count)

# Shuffling the atoms randomly 
random.seed(42) 
random.shuffle(atom_list)

# Building a dummy 1x1x1 BCC unit cell then expand to 3x3x3 (54 empty coordinates)
lattice = Lattice.cubic(3.25)
dummy_struct = Structure(lattice, ["H", "H"], [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]])
dummy_struct.make_supercell([3, 3, 3])

# Swapping the empty coordinates with the shuffled HEA atoms
for i in range(TOTAL_ATOMS):
    dummy_struct.replace(i, atom_list[i])

# Exporting the final blueprint
file_name = f"Optimal_{''.join(elements)}_Blueprint.cif"
CifWriter(dummy_struct).write_file(file_name)

print("\n>>> Finished <<<")
print(f"Total Atoms in Supercell: {len(dummy_struct)}")
print(f"Saved 3D Blueprint as: '{file_name}'")

--- Training model for Refractory specific category ---
Best candidate acquired Score: 9.08 GPa/(g/cm³)
Composition: W:30% - Mo:35% - Ta:5% - Nb:5% - V:25%
Density: 12.06 g/cm³ | Strength: 109.47 GPa

--- Training model for Corrosion specific category ---
Best candidate acquired Score: 11.36 GPa/(g/cm³)
Composition: Co:5% - Cr:35% - Fe:30% - Ni:25% - Cu:5%
Density: 7.92 g/cm³ | Strength: 89.89 GPa

--- Training model for Lightweight specific category ---
Best candidate acquired Score: 9.17 GPa/(g/cm³)
Composition: Al:25% - Mg:25% - Li:10% - Ti:35% - Zn:5%
Density: 3.17 g/cm³ | Strength: 29.13 GPa

Optimization Complete.


In [ ]:
from chgnet.model.model import CHGNet
chgnet = CHGNet.load()
from pymatgen.core import Structure
from chgnet.model import StructOptimizer, CHGNet
import warnings

warnings.filterwarnings("ignore")

print("Loading blueprint and ML model...")

# Loading the HEA structure
struct = Structure.from_file("Optimal_AlTiScZrV_Blueprint.cif")

# Loading CHGNet model
chgnet = CHGNet.load()

# Set up optimizer for CPU 
optimizer = StructOptimizer(model=chgnet, use_device="cpu")

# Relaxing the structure
print("Relaxing structure...")
result = optimizer.relax(struct, fmax=0.05)

# Relaxed structure and final energy
relaxed_struct = result["final_structure"]
energy = result["trajectory"].energies[-1]

print("\n>>> Relaxation Complete <<<")
print(f"Final System Energy: {energy:.4f} eV")

# Saving the blueprint
relaxed_struct.to_file("Relaxed_Optimal_AlTiScZrV_Blueprint.cif")
print("Saved relaxed blueprint as: 'Relaxed_Optimal_AlTiScZrV_Blueprint.cif'")